# XGBoost: Is a better alternative? 

The Mercedes dataset shows some linear relationsships and it appears to respond well to Rigge and Random Forest. However, we notice in the EDA that correlations are not strong and we would try other ensemble method to explore non-linear interactions in the dataset. 

- Goal: We want to compare scores from Ridge Regression, Random Forest and XGboost 

**Note about Ensemble Methods:** 
- Random Forest = ensemble method (bagging) 

- XGBoost = ensemble method (boosting).

- For both, hyperparameter tuning needed to get the best results


In [28]:
# Install xgboost if not already installed
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


**Note for Mac OS** if error:  XGBoost can’t find the OpenMP runtime (libomp.dylib) on Mac. Install libomp using Homebrew 
- brew install libomp

Restart Kernel try to install xgboost again from terminal or from the cell above

In [29]:
import xgboost
print(xgboost.__version__)

3.0.4


# 1. Imports and data processing 
Imports / Droping and transforming columns (OHE)

In [30]:
# imports 
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


# New recommended
from scipy.stats import loguniform, randint, uniform

import matplotlib.pyplot as plt
import matplotlib.cm as cm

## NEW for XGBoost
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


# Load datasets
train_df = pd.read_csv("data/train.csv")
test_df  = pd.read_csv("data/test.csv")

# Drop columns with no variance (from EDA)
drop_cols = ['X11','X93','X107','X233','X235','X268','X289','X290','X293','X297','X330','X347']
train_df = train_df.drop(columns=drop_cols, errors="ignore")
test_df  = test_df.drop(columns=drop_cols, errors="ignore")

# 3) One-Hot Encode categorical columns
categorical_cols = ['X0','X1','X2','X3','X4','X5','X6','X8']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')

# Ohe on train categoricals
ohe_train = ohe.fit_transform(train_df[categorical_cols])
# Ohe on test categoricals
ohe_test  = ohe.transform(test_df[categorical_cols])

# Transformed dataframes (drop original categoricals, join OHE)
train_transformed = train_df.drop(columns=categorical_cols).join(ohe_train)
test_transformed  = test_df.drop(columns=categorical_cols).join(ohe_test)

# Quick checks
print("Shapes")
print("Original train_df shape:    ", train_df.shape)
print("Transformed train_df shape: ", train_transformed.shape)

print("\n Dropped Categorical Columns")
print(categorical_cols)

print("\n New OHE Columns Names")
print(ohe.get_feature_names_out(categorical_cols)[:20])  # show first 20 OHE columns
print(f"Total new OHE columns: {len(ohe.get_feature_names_out(categorical_cols))}")

Shapes
Original train_df shape:     (4209, 366)
Transformed train_df shape:  (4209, 553)

 Dropped Categorical Columns
['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8']

 New OHE Columns Names
['X0_a' 'X0_aa' 'X0_ab' 'X0_ac' 'X0_ad' 'X0_af' 'X0_ai' 'X0_aj' 'X0_ak'
 'X0_al' 'X0_am' 'X0_ao' 'X0_ap' 'X0_aq' 'X0_as' 'X0_at' 'X0_au' 'X0_aw'
 'X0_ax' 'X0_ay']
Total new OHE columns: 195


## 2. Define X and Y

In [31]:
#Define X and y 
feature_cols = [c for c in train_transformed.columns if c not in ['ID', 'y']]
X = train_transformed[feature_cols]
y = train_transformed['y']

# Train-Test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train/valid shapes ->",
      "X_train:", X_train.shape, " X_valid:", X_valid.shape)

Train/valid shapes -> X_train: (3367, 551)  X_valid: (842, 551)


## 3 Baseline XGboost for Hyperparamenter Search

In [ ]:
# (Important to reduce time of training) XGBoost is faster with float32 
X_train_xgb = X_train.astype('float32') # uses less memory when converted to float32
X_valid_xgb = X_valid.astype('float32') 

In [33]:
# Base model for the parameter search (no early stopping yet)
xgb_base = XGBRegressor(
    n_estimators=800,         # Boosted treess - Increase / Decrease with results of baseline
    learning_rate=0.05,       # small step (for now, slower = safer) 
    max_depth=6,              # depth of each tree (interaction complexity)
    min_child_weight=1,       # weight of leaves  (>1 reduces overfit) 
    subsample=0.8,            # row sampling per tree
    colsample_bytree=0.8,     # feature sampling per tree
    reg_lambda=1.0,           # L2 regularization
    reg_alpha=0.0,            # L1 regularization (try >0 with many OHE cols)
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'        # 'gpu_hist' if using GPU 
)

In [34]:
#  Parameter distributions for baseline
#  (to explore "nearby" configs efficiently)
param_dist = {
    # Around max_depth=6
    'max_depth':        randint(4, 9),                # 4–8
    # Around min_child_weight=1
    'min_child_weight': randint(1, 6),                # 1–5
    # Around learning_rate=0.05 (log scale around it)
    'learning_rate':    loguniform(0.02, 0.12),       # ~[0.02, 0.12]
    # Around subsample=0.8 and colsample_bytree=0.8
    'subsample':        uniform(0.7, 0.3),            # [0.7, 1.0]
    'colsample_bytree': uniform(0.7, 0.3),            # [0.7, 1.0]
    # Around reg_lambda=1.0 (allow more/less L2)
    'reg_lambda':       loguniform(0.3, 4.0),         # ~[0.3, 4.0]
    # Around reg_alpha=0.0 (let it try some L1 for sparse OHE)
    'reg_alpha':        loguniform(1e-4, 0.2)         # ~[0.0001, 0.2]
}

In [35]:
# Cross-validation plan
cv = KFold(n_splits=5, shuffle=True, random_state=42)


When looking for parameters for Random Forest / XGboost a randomized search is more efficient that a grid search (we used GS for Ridge regression)

In [36]:
# Randomized Search 
rs = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,                                # 30–100 depending on time
    scoring='neg_root_mean_squared_error',    # optimize RMSE 
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

rs.fit(X_train_xgb, y_train)

print("Best parameters (RandomizedSearchCV):", rs.best_params_)
print("Best CV RMSE (mean across folds):", -rs.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters (RandomizedSearchCV): {'colsample_bytree': np.float64(0.7495800817189007), 'learning_rate': np.float64(0.020568256760865697), 'max_depth': 4, 'min_child_weight': 1, 'reg_alpha': np.float64(0.00045286256843283834), 'reg_lambda': np.float64(0.30432196407793727), 'subsample': np.float64(0.9446384285364502)}
Best CV RMSE (mean across folds): 8.660143609589863


## 4. re-Fit XGboost regressor using results of parameter search

Using the best parameters that we found above train the model again. This time including early stoping in training set (validation split)

In [37]:
# Refit best params with EARLY STOPPING on validation split
best_params = rs.best_params_.copy()

xgb_tuned = XGBRegressor(
    **best_params,
    n_estimators=5000,                 # Increased / early stopping to picks optimal trees
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=-1,
    tree_method='hist',                # 'gpu_hist' if using GPU 
    early_stopping_rounds=100          # (early stoping)stop if no improvement after 100 rounds
)

xgb_tuned.fit(
    X_train_xgb, y_train,
    eval_set=[(X_valid_xgb, y_valid)], # last eval set is monitored
    verbose=False                      # when true prints progress. If set to 100 - it can show progress each 100 rounds 
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,np.float64(0.7495800817189007)
,device,None
,early_stopping_rounds,100
,enable_categorical,False
,eval_metric,'rmse'


## 5. Evaluation 
RMSE is the most common metric but since we want to compare with our previous models (Ridge, Random Forest) we added more metrics

In [38]:


# Evaluate (same metrics as Ridge and RF)
y_pred = xgb_tuned.predict(
    X_valid_xgb,
    iteration_range=(0, xgb_tuned.best_iteration + 1)
)

rmse = float(np.sqrt(mean_squared_error(y_valid, y_pred)))
mae  = float(mean_absolute_error(y_valid, y_pred))
r2   = float(r2_score(y_valid, y_pred))

print("\nXGBoost — Tuned + Early Stopping (Validation)")
print(f"Best trees: {xgb_tuned.best_iteration}")
print(f"RMSE: {rmse:.3f} | MAE: {mae:.3f} | R²: {r2:.3f}")


XGBoost — Tuned + Early Stopping (Validation)
Best trees: 126
RMSE: 7.984 | MAE: 5.312 | R²: 0.590
